> **Daffa was here**

In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

currentRoot = Path.cwd().resolve()
projectRoot = currentRoot.parent if currentRoot.name == 'notebooks' else currentRoot
if not (projectRoot / 'notebooks').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory.')

dataRoot = projectRoot / 'data'
rawRoot = dataRoot / 'raw'
interimRoot = dataRoot / 'interim'
processedRoot = dataRoot / 'processed'
figureRoot = projectRoot / 'reports' / 'figures'
for outputRoot in (rawRoot, interimRoot, processedRoot, figureRoot):
    outputRoot.mkdir(parents=True, exist_ok=True)

configuredRawRoot = os.environ.get('SKIN_DATA_ROOT')
rawCandidates = [Path(configuredRawRoot).expanduser()] if configuredRawRoot else []
rawCandidates.extend([rawRoot, projectRoot.parent / 'Data'])

def containsDatasetFiles(path: Path) -> bool:
    datasetSuffixes = {'.csv', '.zip', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    return path.exists() and any(
        candidate.is_file() and candidate.suffix.lower() in datasetSuffixes
        for candidate in path.rglob('*')
    )

sourceDataRoot = next((path.resolve() for path in rawCandidates if containsDatasetFiles(path)), None)
if sourceDataRoot is None:
    raise FileNotFoundError('No raw dataset found. Populate data/raw or set SKIN_DATA_ROOT.')

def toProjectPath(path: Path) -> str:
    return Path(os.path.relpath(path.resolve(), projectRoot)).as_posix()

def fromProjectPath(pathValue: str) -> Path:
    return (projectRoot / Path(pathValue)).resolve()

print(f'Project root: {projectRoot}')
print(f'Raw dataset root: {sourceDataRoot}')

Project root: D:\sunib\Research\Skin\skin-domain-shift
Raw dataset root: D:\sunib\Research\Skin\Data


In [2]:
clinicalValidatedPath = interimRoot / 'clinicalValidated.csv'
isicValidatedPath = interimRoot / 'isicValidated.csv'
for requiredPath in (clinicalValidatedPath, isicValidatedPath):
    if not requiredPath.exists():
        raise FileNotFoundError(f'Run notebook 02 first; missing {requiredPath.name}.')
clinicalValidated = pd.read_csv(clinicalValidatedPath, low_memory=False)
isicValidated = pd.read_csv(isicValidatedPath, low_memory=False)

In [3]:
clinicalLabelMapping = {'ACK': 'AK', 'BCC': 'BCC', 'MEL': 'MEL', 'NEV': 'NV', 'SCC': 'SCC'}
isicLabelMapping = {'AK': 'AK', 'BCC': 'BCC', 'MEL': 'MEL', 'NV': 'NV', 'SCC': 'SCC'}
strictLabels = ['AK', 'BCC', 'MEL', 'NV', 'SCC']
clinicalSixLabels = ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']

coreColumns = [
    'imageId', 'imagePath', 'sourceDataset', 'originalLabel', 'harmonizedLabel',
    'patientId', 'lesionId', 'split', 'imageWidth', 'imageHeight', 'imageFormat',
    'metadataAvailable', 'fileSize', 'modifiedNs', 'isValid',
]

clinicalEligible = clinicalValidated.loc[
    clinicalValidated['isValid'].astype(bool) & clinicalValidated['originalLabel'].isin(clinicalSixLabels)
].copy()
clinicalSixClass = clinicalEligible.copy()
clinicalSixClass['harmonizedLabel'] = clinicalSixClass['originalLabel'].map(
    {**clinicalLabelMapping, 'SEK': 'SEK'}
)
clinicalFiveClass = clinicalEligible.loc[clinicalEligible['originalLabel'].isin(clinicalLabelMapping)].copy()
clinicalFiveClass['harmonizedLabel'] = clinicalFiveClass['originalLabel'].map(clinicalLabelMapping)

isicEligible = isicValidated.loc[
    isicValidated['isValid'].astype(bool) & isicValidated['originalLabel'].notna()
].copy()
isicFiveClass = isicEligible.loc[isicEligible['originalLabel'].isin(isicLabelMapping)].copy()
isicFiveClass['harmonizedLabel'] = isicFiveClass['originalLabel'].map(isicLabelMapping)

for frame in (clinicalSixClass, clinicalFiveClass, isicFiveClass):
    for column in coreColumns:
        if column not in frame.columns:
            frame[column] = pd.NA

clinicalSixClassPath = processedRoot / 'clinicalSixClass.csv'
clinicalFiveClassPath = processedRoot / 'clinicalFiveClass.csv'
isicFiveClassPath = processedRoot / 'isicFiveClass.csv'
clinicalSixClass.to_csv(clinicalSixClassPath, index=False)
clinicalFiveClass.to_csv(clinicalFiveClassPath, index=False)
isicFiveClass.to_csv(isicFiveClassPath, index=False)

print('Clinical six-class counts:')
print(clinicalSixClass['originalLabel'].value_counts().sort_index())
print('\nClinical harmonized five-class counts:')
print(clinicalFiveClass['harmonizedLabel'].value_counts().sort_index())
print(f'\nISIC harmonized labelled rows: {len(isicFiveClass):,}')

Clinical six-class counts:
originalLabel
ACK    730
BCC    845
MEL     52
NEV    244
SCC    192
SEK    235
Name: count, dtype: int64

Clinical harmonized five-class counts:
harmonizedLabel
AK     730
BCC    845
MEL     52
NV     244
SCC    192
Name: count, dtype: int64

ISIC harmonized labelled rows: 27,551


In [4]:
exclusionFrames = []

invalidClinical = clinicalValidated.loc[~clinicalValidated['isValid'].astype(bool), ['imageId', 'imagePath', 'sourceDataset', 'originalLabel']].copy()
invalidClinical['exclusionStage'] = 'labelHarmonization'
invalidClinical['exclusionReason'] = 'Failed data validation'
exclusionFrames.append(invalidClinical)

clinicalSek = clinicalEligible.loc[clinicalEligible['originalLabel'].eq('SEK'), ['imageId', 'imagePath', 'sourceDataset', 'originalLabel']].copy()
clinicalSek['exclusionStage'] = 'strictFiveClass'
clinicalSek['exclusionReason'] = 'SEK has no strict one-to-one ISIC mapping; BKL is broader'
exclusionFrames.append(clinicalSek)

isicNonOverlap = isicEligible.loc[~isicEligible['originalLabel'].isin(isicLabelMapping), ['imageId', 'imagePath', 'sourceDataset', 'originalLabel']].copy()
isicNonOverlap['exclusionStage'] = 'strictFiveClass'
isicNonOverlap['exclusionReason'] = 'ISIC category is outside the strict shared five-class taxonomy'
exclusionFrames.append(isicNonOverlap)

isicUnlabelled = isicValidated.loc[isicValidated['originalLabel'].isna(), ['imageId', 'imagePath', 'sourceDataset', 'originalLabel']].copy()
isicUnlabelled['exclusionStage'] = 'strictFiveClass'
isicUnlabelled['exclusionReason'] = 'Official ISIC ground truth is unavailable'
exclusionFrames.append(isicUnlabelled)

excludedSamples = pd.concat(exclusionFrames, ignore_index=True)
excludedSamplesPath = processedRoot / 'excludedSamples.csv'
excludedSamples.to_csv(excludedSamplesPath, index=False)

labelMapping = {
    'strictClasses': strictLabels,
    'clinicalMapping': clinicalLabelMapping,
    'isicMapping': isicLabelMapping,
    'unmappedClinicalLabels': {
        'SEK': 'Retained in clinical six-class data; excluded from strict cross-domain data because BKL is broader.'
    },
    'clinicalSixClassCount': len(clinicalSixClass),
    'clinicalFiveClassCount': len(clinicalFiveClass),
    'isicFiveClassCount': len(isicFiveClass),
    'excludedCount': len(excludedSamples),
}
labelMappingPath = processedRoot / 'labelMapping.json'
labelMappingPath.write_text(json.dumps(labelMapping, indent=2), encoding='utf-8')
excludedSamples['exclusionReason'].value_counts()

exclusionReason
ISIC category is outside the strict shared five-class taxonomy    6018
SEK has no strict one-to-one ISIC mapping; BKL is broader          235
Name: count, dtype: int64

In [5]:
assert clinicalLabelMapping == {'ACK': 'AK', 'BCC': 'BCC', 'MEL': 'MEL', 'NEV': 'NV', 'SCC': 'SCC'}
assert 'SEK' not in clinicalLabelMapping
assert set(clinicalFiveClass['harmonizedLabel'].dropna().unique()).issubset(set(strictLabels))
assert len(clinicalSixClass) == len(clinicalFiveClass) + int(clinicalSixClass['originalLabel'].eq('SEK').sum())
print('Label harmonization checks passed.')

Label harmonization checks passed.
